<a href="https://colab.research.google.com/github/davidrpugh/data-analytics-with-polars/blob/main/ch10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 10: Selecting and Creating Columns

This notebook accompanies Chapter 10 of *Python Polars: The Definitive Guide* by Jeroen Janssens and Thijs Nieuwdorp. It is adapted for **Polars 2.1** and **Google Colab**.

Part IV of the book puts expressions to work **transforming DataFrames**. We start with columns, because selecting existing columns and creating new ones are probably the most common things you do with data.

By the end of this notebook you will be able to:

- select columns with `select`, and by **name**, **data type**, or **position** using **column selectors** (`polars.selectors`, imported as `cs`)
- **combine** selectors with set operations (union, intersection, difference, ...)
- create columns with `with_columns`, and explain why the expressions in one call **can't depend on each other**
- drop, rename, and stack columns, and know when stacking by position is risky
- add a row index

Our data is the Star Wars characters table. We focus on three rebels.

**How to use this notebook:** run the cells from top to bottom. Some cells *intentionally* trigger errors; those are caught and printed, so they won't stop the notebook.

## 0. Setup

We pin Polars to the **2.1 series** so everyone sees the same behavior, and use `%pip` so the package is installed into the running notebook kernel.

In [ ]:
%pip install -q "polars==2.1.*"

### Imports and helpers

- `download` fetches a data file from the book's repository **only if it isn't already there**. It writes to a temporary name and renames when complete, and raises on HTTP errors instead of saving an error page as if it were data.

By convention, the selectors module is imported as `cs`, for *column selectors*.

In [ ]:
from pathlib import Path
import urllib.request

import polars as pl
import polars.selectors as cs
from polars.testing import assert_frame_equal


print(f"Polars {pl.__version__}")


BASE_URL = "https://github.com/davidrpugh/data-analytics-with-polars/raw/refs/heads/main/data"
DATA_DIR = Path("data")


def download(relative_path: str) -> Path:
    """Download BASE_URL/relative_path into DATA_DIR (unless it exists) and return the local path."""
    path = DATA_DIR / relative_path
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary = path.with_name(path.name + ".tmp")
        urllib.request.urlretrieve(f"{BASE_URL}/{relative_path}", temporary)
        temporary.rename(path)
    return path

### The data

The `starwars` table has one row per character. We keep three rebels and drop the `films` column (a list of films), which would only clutter the output. The result has 15 columns of many different types, which is what we need to demonstrate selecting by type.

Instead of printing slices of the table with brackets to fit the page (the book's workaround for a wide table), we use `glimpse()`, which prints one line per column: its name, type, and first values.

Two columns need explanation. `birth_date` is a made-up column: a Gregorian-calendar date, assuming year 0 BBY started on May 25, 1977 (the release date of *A New Hope*). `screen_time` is the total on-screen time in the original trilogy.

In [ ]:
STARWARS_PARQUET = download("starwars.parquet")

starwars = pl.read_parquet(STARWARS_PARQUET)

rebels = starwars.drop("films").filter(
    pl.col("name").is_in(["Luke Skywalker", "Leia Organa", "Han Solo"])
)

rebels.glimpse()

## Selecting Columns

`select` keeps the columns you describe, *in the order you list them*, and drops the rest. You can select one column, several, or all; the same column more than once (if the names don't clash, because column names must be unique); and you can **create** new columns in the same call.

You've seen three ways to describe a column: a **string** (shortest, but a string isn't an expression, so you can't do math on it), `pl.col(...)` (an expression, so you can customize it), and `pl.col` with a **regex** (a pattern that starts with `^` and ends with `$`, matching many column names at once). Here are all three in one call, plus arithmetic:

In [ ]:
rebels.select(
    "name",
    pl.col("homeworld"),
    pl.col("^.*_color$"),
    (pl.col("height") / 100).alias("height_m"),
)

## Introducing Selectors

**Selectors** are a third, more flexible way to describe columns, by their **name**, **data type**, or **position**, and they can be **combined**. A selector is an expression under the hood, so you can use it anywhere you'd use `pl.col(...)`, and anywhere columns are specified: `select`, `with_columns`, `filter`, `group_by`, `agg`, and more.

The closest equivalents of `pl.col` are `cs.by_name` (exact names) and `cs.matches` (a regex). On their own they aren't more convenient than `pl.col`, which is why the next sections introduce the selectors that are.

In [ ]:
rebels.select(
    "name",
    cs.by_name("homeworld"),
    cs.matches("^.*_color$"),
    (cs.by_name("height") / 100).alias("height_m"),
)

### Selecting based on name

Instead of writing a regex, use a selector that says what you mean:

| Selector | Selects columns whose name... |
|---|---|
| `cs.starts_with(...)` | starts with the text |
| `cs.ends_with(...)` | ends with the text |
| `cs.contains(...)` | contains the text |
| `cs.matches(...)` | matches a regex |
| `cs.by_name(...)` | is one of the exact names given |
| `cs.alpha()`, `cs.alphanumeric()`, `cs.digit()` | consists only of letters / letters and digits / digits |

In [ ]:
rebels.select(cs.starts_with("birth_"))

In [ ]:
rebels.select(cs.ends_with("_color"))

In [ ]:
rebels.select(cs.contains("_"))

A regex gives you full control. For instance, names of exactly four lowercase letters:

In [ ]:
rebels.select(cs.matches("^[a-z]{4}$"))

The last three name-based selectors are about the *characters* in a name. We use a small DataFrame with deliberately odd names, and `cs.expand_selector` to see which columns a selector would pick, without running a query:

In [ ]:
names_df = pl.DataFrame({"total": [1], "col2": [2], "col_3": [3], "2024": [4]})

for selector in [cs.alpha(), cs.alphanumeric(), cs.digit()]:
    print(f"{str(selector):45} => {cs.expand_selector(names_df, selector)}")

**Missing names.** What if you ask `cs.by_name` for a column that doesn't exist? Newer versions of Polars raise an error by default, because a typo in a column name should be loud. Pass `require_all=False` to allow missing names and select only what exists (check the documentation for the exact behavior in your version):

In [ ]:
rebels.select(cs.by_name("name", "not_a_column")).columns

In [ ]:
rebels.select(cs.by_name("name", "not_a_column", require_all=False)).columns

### Selecting based on data type

Sometimes you care about a column's **type** more than its name. For instance, to average *every numeric column* for each hair color, without listing them. The `group_by` is followed by a `sort`, because Polars doesn't guarantee the order of the groups:

In [ ]:
rebels.group_by("hair_color").agg(cs.numeric().mean()).sort("hair_color")

`cs.numeric()` covers integers *and* floats. The type selectors:

| Selector | Selects |
|---|---|
| `cs.numeric()` | all numbers |
| `cs.integer()`, `cs.signed_integer()`, `cs.unsigned_integer()`, `cs.float()`, `cs.decimal()` | one kind of number |
| `cs.string()`, `cs.categorical()`, `cs.boolean()`, `cs.binary()` | those types |
| `cs.temporal()` | all of date, time, datetime, duration |
| `cs.date()`, `cs.time()`, `cs.datetime(...)`, `cs.duration(...)` | one temporal type (optionally filtered by unit or time zone) |
| `cs.by_dtype(...)` | exactly the data types you list |

In [ ]:
rebels.select(cs.string())

In [ ]:
rebels.select(cs.temporal())

Nested types need the *inner* type too, so use `cs.by_dtype` with the full type, such as a List of Strings:

In [ ]:
rebels.select(cs.by_dtype(pl.List(pl.String)))

### Selecting based on position

The third kind selects by **position** (`cs.by_index`, `cs.first`, `cs.last`). There are few good reasons to rely on a column's position, because a position changes when someone adds a column. An occasional use: a DataFrame with many columns whose names don't help. For example, every third column, using a `range`.

Newer Polars versions raise an error if an index is out of bounds, so we pass `require_all=False`, which tells Polars to skip indices that don't exist (the range goes far beyond the number of columns on purpose). Check the documentation for this argument in your version.

In [ ]:
rebels.select(cs.by_index(range(0, 999, 3), require_all=False))

Negative positions count from the right. A natural use: right after adding columns, which land at the **end**, select the first column plus the last two:

In [ ]:
rebels.select("name", cs.by_index(range(-2, 0)))

An out-of-range index is an error. A `range` that is completely out of bounds, on the other hand, may quietly give an empty DataFrame, depending on your version. Always check what yours does:

In [ ]:
rebels.select(cs.by_index(20))

In [ ]:
rebels.select(cs.by_index(range(20, 22), require_all=False)).shape

### Combining selectors

This is where selectors become more powerful than `pl.col`: you can combine them with **set operations**. The book points out that you can't say "the column `hair_color` *and* all numeric columns" with `pl.col`. With selectors it's one expression:

In [ ]:
rebels.select(cs.by_name("hair_color") | cs.numeric())

The columns come back in the order they appear in the DataFrame, not in the order you wrote the selectors.

There are five set operators, which behave like operations on sets of column names:

| Operation | Operator | Selects |
|---|---|---|
| Union | `x \| y` | columns in x, or y, or both |
| Intersection | `x & y` | columns in *both* x and y |
| Difference | `x - y` | columns in x that are *not* in y |
| Exclusive or | `x ^ y` | columns in x or y, but not both |
| Complement | `~x` | all columns *not* in x |

To see them in action without guessing, we use a tiny DataFrame, where `x` selects three columns by name and `y` selects the Boolean columns. `cs.expand_selector` tells us which columns each combination picks. (The book does this with `eval` on strings; we build a dictionary of the selectors themselves, which is safer and easier to read.)

In [ ]:
selector_demo_df = pl.DataFrame({"d": 1, "i": True, "s": True, "c": True, "o": 1.0})
print(selector_demo_df.schema)

x = cs.by_name("d", "i", "s")
y = cs.boolean()

operations = {
    "x": x,
    "y": y,
    "x | y": x | y,
    "x & y": x & y,
    "x - y": x - y,
    "x ^ y": x ^ y,
    "~x": ~x,
    "x - x": x - x,
}

pl.DataFrame(
    {
        "selector": list(operations),
        "columns": [", ".join(cs.expand_selector(selector_demo_df, s)) for s in operations.values()],
    }
)

The last row shows that you can end up with **no columns at all** (`x - x` removes everything), which gives an empty DataFrame:

In [ ]:
selector_demo_df.select(x - x)

**Moving columns to the front.** A neat trick uses Python's *walrus operator* (`:=`), which assigns a value and uses it in the same expression. Assign a selector to a name in the first argument, then reuse it, negated (`~`), in the second argument: "these columns first, then all the others". Here `c` and `i` move to the front:

In [ ]:
selector_demo_df.select(front := cs.by_name("c", "i"), ~front)

This is especially handy right after you create a column, because new columns land at the *end*. To move the last column to the front, use `cs.last()` the same way. The book calls this variable `first`, which is confusing next to `cs.first()`; we call it `front`.

In [ ]:
selector_demo_df.select(front := cs.last(), ~front)

## Creating Columns

`with_columns` **adds** columns to the right of the existing ones, keeping everything that was already there. For example, the **body mass index** of each rebel from `mass` (kg) and `height` (cm). We define the formula once as a named expression (an expression is a reusable object, Chapter 7) and give the new column a name with a keyword:

In [ ]:
bmi = pl.col("mass") / ((pl.col("height") / 100) ** 2)

rebels.with_columns(bmi=bmi).select("name", "height", "mass", "bmi")

### Beware: a column with an existing name is overwritten

If the new column has the **same name** as an existing one, the existing one is **replaced**. That's a common accident:

In [ ]:
a_df = pl.DataFrame({"a": [1, 2, 3]})
a_df.with_columns(pl.col("a") * 2)

Give the new column its own name to keep the original:

In [ ]:
a_df.with_columns(a2=pl.col("a") * 2)

### Several columns in one call

You can create many columns at once. Here we also compute how old each rebel was when the second Death Star was destroyed (May 25, 1983). Date subtraction gives a *duration*; `dt.total_days()` turns it into a number of days, and integer division by 365 gives approximate whole years. (It's approximate: leap years make 365 slightly off.)

In [ ]:
rebels.with_columns(
    bmi=bmi,
    age_destroy=((pl.date(1983, 5, 25) - pl.col("birth_date")).dt.total_days() // 365).cast(pl.UInt8),
).select("name", "birth_date", "bmi", "age_destroy")

### Expressions in the same call can't depend on each other

All the expressions in a `select` or `with_columns` call are evaluated **in parallel**, each starting from the *original* DataFrame. That's what makes Polars fast, but it means an expression can't use a column that is *created in the same call*. Here `bmi_cat` tries to use the `bmi` column that is being created next to it. We expect a "column not found" error:

In [ ]:
rebels.with_columns(
        bmi=bmi,
        bmi_cat=pl.col("bmi").bin_intervals([18.5, 25], labels=["Underweight", "Normal", "Overweight"]),
)

There are two fixes.

**Fix 1: chain two `with_columns` calls.** The second call sees the column made by the first. This is the usual solution.

**Fix 2: reuse the expression.** Because `bmi` is an expression *object*, you can build the category directly from it. No second step is needed, at the cost of computing the BMI formula twice (which doesn't matter for a cheap formula, but would for an expensive one).

(The `cut` method needs one more label than breakpoints: two breakpoints make three categories.)

In [ ]:
labels = ["Underweight", "Normal", "Overweight"]

chained = rebels.with_columns(bmi=bmi).with_columns(bmi_cat=pl.col("bmi").bin_intervals([18.5, 25], labels=labels))
reused = rebels.with_columns(bmi=bmi, bmi_cat=bmi.bin_intervals([18.5, 25], labels=labels))


In [ ]:
assert_frame_equal(chained, reused)

In [ ]:
print(reused.schema["bmi_cat"])

In [ ]:
reused.select("name", "bmi", "bmi_cat")

### Creating columns with `select`, and where new columns end up

`select` can create columns too, and lets you choose **which** columns to keep and **where** the new column goes. `with_columns(...)` is shorthand for `select(pl.all(), ...)`, but clearer and shorter. We check that the two agree:

In [ ]:
with_columns_version = a_df.with_columns(pl.lit(1).alias("ones"))
select_version = a_df.select(pl.all(), pl.lit(1).alias("ones"))

assert_frame_equal(with_columns_version, select_version)
with_columns_version

**Naming a column in the middle.** Python requires **keyword arguments to come after positional ones**. So if you want a *named* new column between two existing columns, you can't use the keyword form: `select("name", bmi=..., "species")` is not even valid Python.

In [ ]:
starwars.select(
    "name",
    bmi=(pl.col("mass") / ((pl.col("height") / 100) ** 2)),
    "species",
)


The fix is `alias`, which attaches the name to the expression itself, so it can sit anywhere in the argument list. Here we find the five characters with the highest BMI. Two details:

- `drop_nulls("bmi")` only drops rows where the **BMI** is missing (not rows with any missing value, as the book's version does)
- `top_k` returns the top rows in **no guaranteed order**, so we sort afterwards

In [ ]:
(
    starwars.select("name", bmi.alias("bmi"), "species")
            .drop_nulls("bmi")
            .top_k(5, by="bmi")
            .sort("bmi", descending=True)
)

Jabba the Hutt is a big outlier. That's a reminder that a formula designed for humans doesn't suit every species.

## Related Column Operations

### Dropping

Sometimes it's easier to say what you *don't* want. `drop` takes column names. By default it is **strict**: asking to drop a column that doesn't exist is an error (a typo shouldn't pass silently). With `strict=False` missing columns are ignored. We already dropped `films` from `rebels`, so:

In [ ]:
rebels.drop("films").shape

In [ ]:
rebels.drop("name", "films", "screen_time", strict=False).columns

Two alternatives with selectors, which also let you drop by type or pattern: negate a selector with `~`, or use `cs.exclude`. We check they agree with `drop`:

In [ ]:
by_drop = rebels.drop("name", "screen_time")
by_negation = rebels.select(~cs.by_name("name", "screen_time"))
by_exclude = rebels.select(cs.exclude("name", "screen_time"))


In [ ]:
assert_frame_equal(by_drop, by_negation)

In [ ]:
assert_frame_equal(by_drop, by_exclude)

In [ ]:
by_drop.columns

`drop` itself also accepts selectors.

In [ ]:
rebels.drop(cs.string(), cs.temporal()).columns

### Renaming

You could rename with `select`, but then you must list every column you want to keep. `rename` is better. It accepts either a **dictionary** of old and new names, or a **function** applied to every name. Here we chain both: first rename two columns explicitly, then strip the `_color` suffix from every name that has it:

In [ ]:
(
    rebels.rename({"homeworld": "planet", "mass": "weight"})
          .rename(lambda name: name.removesuffix("_color"))
          .select("name", "planet", "weight", "hair", "skin", "eye")
)

When you only want to rename the columns *inside one expression* (for example the output of a selector), use the `name` namespace from Chapter 7 (`name.prefix`, `name.suffix`, `name.map`, ...).

In [ ]:
rebels.select(
    cs.ends_with("_color").name.map(lambda name: name.removesuffix("_color"))
)

### Stacking

If you have another DataFrame, or some Series, with the **same number of rows**, `hstack` ("horizontal stack") places them to the right. It matches rows **by position**, not by any key, so it's only correct if both sides are in the same order.

In [ ]:
rebel_names = rebels.select("name")
rebel_colors = rebels.select(cs.ends_with("_color"))
rebel_quotes = pl.Series(
    "quote",
    [
        "You know, sometimes I amaze myself.",
        "That doesn't sound too hard.",
        "I have a bad feeling about this.",
    ],
)

rebel_names.hstack(rebel_colors).hstack([rebel_quotes])

The quotes were written in the order the three rebels appear *today*. That's the weakness of stacking: if the DataFrame is re-sorted, or the source file lists the characters differently, the quotes silently attach to the **wrong people**. Let's prove it by sorting the names in the other direction and stacking the same quotes again. Compare each row's name and quote with the table above:

In [ ]:
rebels.sort("name", descending=True).select("name").hstack([rebel_quotes])

The quotes didn't move, but the names did, so they're now mismatched. The robust way to add data to a DataFrame is a **join on a key** (here, the character's name). `validate="1:1"` states that each name should appear exactly once on both sides, so Polars raises an error if that's ever false (Chapter 14 covers joins):

In [ ]:
quotes_df = pl.DataFrame({"name": rebel_names["name"], "quote": rebel_quotes})

joined = rebels.sort("name", descending=True).select("name").join(quotes_df, on="name", validate="1:1")


In [ ]:
assert joined.sort("name").equals(quotes_df.sort("name"))

In [ ]:
joined

Whichever order the rebels are in, each one keeps their own quote. **Use `hstack` only when the order is under your control. Use a join when rows are identified by a key.** (The same applies to `with_columns` with a Series, and to `pl.concat(..., how="horizontal")`, which also match by position.)

### Adding row indices

`with_row_index` adds a column of consecutive integers at the **front**. You can choose its name and its starting value (`offset`). The index is just a column: there's no special `Index` object like in pandas.

In [ ]:
rebels.with_row_index(name="rebel_id", offset=1).select("rebel_id", "name")

## Takeaways

- `select` keeps the columns you describe, in the order you list them, and can create new ones.
- A column can be described by a **string**, `pl.col(...)` (optionally with a regex), or a **selector**.
- **Selectors** choose columns by **name** (`starts_with`, `ends_with`, `contains`, `matches`, `by_name`), **type** (`numeric`, `string`, `temporal`, `by_dtype`, ...), or **position** (`by_index`, `first`, `last`). Prefer name and type over position.
- Selectors combine with set operations: `|`, `&`, `-`, `^`, and `~`. `cs.expand_selector` shows which columns a selector picks.
- Newer Polars versions are **strict about missing names and out-of-range positions**. Use `require_all=False` when you really mean "whatever exists".
- `with_columns` appends columns (replacing any with the same name). Expressions in one call run in parallel and **can't see each other's results**: chain a second `with_columns`, or reuse the expression.
- Python requires keyword arguments after positional ones. Use `alias` to place a named column anywhere.
- `drop` is strict by default; `rename` accepts a dictionary or a function.
- `hstack` matches rows **by position**. For data identified by a key, use a **join** with `validate=`.
- `with_columns(...)` is shorthand for `select(pl.all(), ...)`.

### Next up

In Chapter 11 we turn from columns to **rows**: filtering and sorting.

### Try it yourself

1. Select every column of `rebels` whose name contains `"_"` **and** whose type is a String, with one selector expression.

2. Add a column `height_m` to `rebels` and move it to the front with the walrus trick.

3. Compute `bmi_cat` for **all** of `starwars` and count the characters in each category. (Remember that `cut` returns a categorical column.)

4. Select all numeric columns *except* `birth_year`, in two ways: with a set operation, and with `cs.exclude`.

5. Rename every column of `rebels` to upper case with one `rename` call.